# PolyPythia seed-replication study

Coupling-Phase Spectroscopy — governed Colab runner.

Runs a common checkpoint and probe contract across PolyPythia seeds. Set `CPS_SEEDS` and `CPS_REVISION` in the environment. Each seed writes an independent immutable manifest.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[pythia,notebooks]"], check=True)
print("repo", repo, "ref", GIT_REF)

In [ ]:
import dataclasses, os
from cps.pythia.config import load_probe_config
from cps.pythia.runner import run_probe
base = load_probe_config("subjects/pythia/configs/polypythia_70m_seed_study.yaml")
seeds = [int(x) for x in os.environ.get("CPS_SEEDS", "1,2").split(",")]
revision = os.environ.get("CPS_REVISION", "step1000")
outputs=[]
for seed in seeds:
    model = dataclasses.replace(base.model, run=f"polypythia-70m-seed{seed}", revision=revision)
    output = dataclasses.replace(base.output, run_name=f"polypythia-70m-seed{seed}")
    cfg = dataclasses.replace(base, model=model, output=output)
    outputs.append(str(run_probe(cfg)))
print("\n".join(outputs))

In [ ]:
import pathlib, shutil
export_dir = pathlib.Path("/content/cps-export")
export_dir.mkdir(parents=True, exist_ok=True)
source = pathlib.Path("/content/cps-artifacts")
if source.exists():
    shutil.copytree(source, export_dir / "artifacts", dirs_exist_ok=True)
shutil.make_archive("/content/cps-export", "zip", "/content/cps-export")
print("exported", export_dir)